# ClaimShield: training and evaluation on a free Colab T4

This notebook rebuilds the whole project from scratch:

1. **Download** 1,653 real room photos (MIT Indoor-67).
2. **Create the dataset.** Each room gets an honest copy and an edited copy, with paint, splice or Stable Diffusion damage.
3. **Train** three patch detectors and keep the best on validation.
4. **Evaluate** on held-out rooms, against baselines and on editing tools the model never saw.

**Before you start:** go to *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The full run takes about 1.5 hours; keep this tab open.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/Kaushik-verse/claimshield.git"   # public repo; change if renamed

import os
if not os.path.isdir("/content/claimshield"):
    !git clone -q {REPO} /content/claimshield
%cd /content/claimshield
# versions that ran cleanly on Colab (newer transformers releases break SD 1.5 loading)
!pip -q install diffusers==0.35.1 peft==0.17.1 transformers==4.57.1 accelerate opencv-python-headless 2>&1 | grep -v "dependency\|gradio\|ERROR: pip" || true
!pip -q install -e . --no-deps

## 1 · Real room photos

A seeded sample of up to 400 photos per room type (bathrooms and dining rooms have fewer). The same seed always gives the same photos.

In [ ]:
!python scripts/fetch_bases.py --per-room 400

## 2 · Build the dataset (~30 min)

For every room photo:
- **honest copy:** phone-style tone tweaks only;
- **edited copy:** one of
  - **paint:** drawn cracks or stains;
  - **splice:** a region pasted from this or another photo;
  - **diffusion:** Stable Diffusion 1.5 inpainting of a damage prompt (guidance 1.8, 8 LCM steps).

An edit must visibly change at least 0.4% of the photo, otherwise it is retried with a new seed.

Both copies then go through the same "delivery": random downscale, JPEG, and sometimes a second messaging-app JPEG.

Two test-only sets measure generalisation: OpenCV inpainting, and 80 fully AI-generated room photos.

In [ ]:
!python scripts/make_dataset.py --diffusion --guidance 1.8 --steps 8 --min-visible 0.004 --ood 80 2>&1 | grep -vE "Warning|absl|cudart|skip_prk"

In [ ]:
# What the edits look like: original | edited | edit mask
import csv, matplotlib.pyplot as plt
from PIL import Image

rows = list(csv.DictReader(open("data/manifest.csv")))
picks = []
for fam in ["paint", "splice", "diffusion"]:
    picks.append(next(r for r in rows if r["family"] == fam and r["split"] == "test"))

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax_row, r in zip(axes, picks):
    for ax, path, title in zip(ax_row,
                               [f"data/images/{r['base_id']}_real.png", "data/" + r["image"], "data/" + r["mask"]],
                               ["original", f"{r['family']} {r['prompt']}".strip(), "edited pixels"]):
        ax.imshow(Image.open(path), cmap="gray"); ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 3 · Train three candidate detectors (~40 min)

Each is an ImageNet backbone fine-tuned on 128 px windows ("were these pixels edited?").

- Every training photo also gets one **laundered** copy (re-saved JPEG, downscaled or screenshotted), so recompression is not mistaken for editing.
- Each epoch is scored on the **validation** rooms, and the best epoch is kept.

In [ ]:
!python scripts/train_patchnet.py --arch resnet18        --epochs 20 --launder --neg-keep 0.25           --out models/cand_resnet18.pt

In [ ]:
!python scripts/train_patchnet.py --arch efficientnet_b0 --epochs 20 --launder --neg-keep 0.25 --lr 1e-3 --out models/cand_effb0.pt

In [ ]:
!python scripts/train_patchnet.py --arch resnet50        --epochs 20 --launder --neg-keep 0.25           --out models/cand_resnet50.pt

## 4 · Pick the winner on validation

Only validation AUC is used here. The test set is touched for the first time in step 6.

In [ ]:
!python scripts/select_model.py models/cand_*.pt

## 5 · Leave-one-family-out (~15 min)

Can the detector catch an editing tool it never trained on? Retrain with one edit type removed, then test on that type.

In [ ]:
import json
ARCH = json.load(open("models/patchnet.json"))["arch"]
LR = "1e-3" if ARCH == "efficientnet_b0" else "3e-4"
for fam in ["paint", "splice", "diffusion"]:
    !python scripts/train_patchnet.py --arch {ARCH} --lr {LR} --epochs 10 --launder --neg-keep 0.25 --exclude {fam} --out models/lofo_{fam}.pt

## 6 · Evaluate on the held-out test rooms (~15 min)

Steps:
- Set the threshold on validation so at most 5% of honest photos are flagged, then freeze it.
- Compare against two classical baselines: ELA and SRM noise residuals.
- Report 95% bootstrap intervals.
- Test robustness to extra JPEG, downscaling and screenshots.

In [ ]:
!mkdir -p results
!python scripts/evaluate.py > results/eval_stdout.json
!python scripts/make_figures.py

In [ ]:
import json, pandas as pd
m = json.load(open("results/metrics.json"))
det = m["detectors"]

overall = pd.DataFrame({k: {"AUC": v["test"]["auc"], "95% CI": v["test"]["auc_95ci"],
                            "caught": v["test"]["tpr"], "honest flagged": v["test"]["fpr"]}
                        for k, v in det.items()}).T
print("Held-out test set"); display(overall)

pn = det["patchnet"]
by_type = pd.DataFrame({**{f: {"AUC": s["auc"], "caught": s["tpr"]} for f, s in pn["per_family"].items()},
                        **{f + " (unseen)": {"AUC": s["auc"], "caught": s["tpr"]} for f, s in pn["unseen_tools"].items()}}).T
print("Patch net by edit type"); display(by_type)

print("Robustness"); display(pd.DataFrame(pn["robustness"]).T[["auc", "tpr", "fpr"]])
print("Leave-one-family-out"); display(pd.DataFrame({f: {k: v["auc"] for k, v in e.items()} for f, e in m["lofo"].items()}).T)

In [ ]:
# Heatmaps on a few test photos
from claimshield.cli import load_model, overlay
from claimshield.patchnet import heatmap
import numpy as np

model, thr = load_model()
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, fam in zip(axes.flat, ["authentic", "paint", "splice", "diffusion", "telea_unseen", "fullgen"]):
    r = next(r for r in rows if r["family"] == fam and r["split"] in ("test", "ood"))
    img = np.asarray(Image.open("data/" + r["image"]).convert("RGB"))
    hm, score = heatmap(model, img)
    ax.imshow(overlay(img, hm)); ax.axis("off")
    ax.set_title(f"{fam}: score {score:.2f} → {'FLAG' if score > thr else 'ok'}", fontsize=9)
plt.tight_layout(); plt.show()

## 7 · Tests and download

In [ ]:
!python -m pytest -q tests

In [ ]:
!tar czf claimshield_results.tar.gz results models/*.pt models/*.json data/manifest.csv data/bases_manifest.csv data/generation_stats.json docs/results.svg
from google.colab import files
files.download("claimshield_results.tar.gz")